## action def

This notebook introduces `action def`; after running it you can declare a named action with typed inputs and outputs, an ordered sequence of sub-actions, and an assignment that wires a calculation into the flow.

The Chapter 3 model expresses *what* the toaster must accomplish (the requirement) and *how much* energy it delivers (the calculation). Chapter 4 adds the functional layer: *how* the system transforms inputs into outputs step by step. `action def` in SysML v2 declares a named behavior with `in`/`out` parameters, a `first`/`then` sequence, and nested `action` steps. This notebook adds `ApplyHeat` to the model.

In [ ]:
import opensysml
from toaster.report import format_diagnostics

source = """package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }
    part def Heater { attribute power : Real default = 800.0; }
    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;
    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
    part nominal : Toaster;
    part slow : Toaster { attribute :>> cycleTime = 200.0; }
    requirement def TimelyToast {
        subject toaster : Toaster;
        require constraint { toaster.cycleTime <= 180.0 }
    }
    requirement timely : TimelyToast;
    part evidence {
        assert satisfy timely by nominal;
        assert satisfy timely by slow;
    }
    calc def DeliveredEnergy {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        return : Real = power * duration * efficiency;
    }
    action def ApplyHeat {
        in power : Real;
        in duration : Real;
        in efficiency : Real;
        out energy : Real;
        first start;
        then action calculate {
            assign energy := DeliveredEnergy(power, duration, efficiency);
        }
        then done;
    }
}"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: an action def that references an undefined calculation
# in an assign statement raises "unresolved reference" at that site.
bad_source = """
package Bad {
    private import ScalarValues::*;
    action def BadAction {
        in power : Real;
        out energy : Real;
        first start;
        then action step { assign energy := UndefinedCalc(power); }
        then done;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
action = model.find("ToasterDemo::ApplyHeat")
assert action is not None
print(f"action kind: {action.kind}")
print(f"action id  : {action.id}")
conn.close()

`action def ApplyHeat { in power : Real; ... first start; then action calculate { ... } then done; }` is the A-F declaration; OpenSysML parses the sequence and sub-action assignments (O-S); `model.find()` returns the ActionDefinition symbol (E).

Try the chapter exercise in `exercises/ch04/exercise.ipynb`: declare an `EjectToast` action def with appropriate inputs and a `first`/`then` sequence for the bread-removal path.